# Lab 11 — Evaluation & LLM-as-Judge

**Difficulty:** ★☆☆ · **optional · GAP LAB** · Est. build 1 day

**Watch / read first**
- Lecture 12 (Evaluations) — https://rlhfbook.com/course/
- Chapter 16 — https://rlhfbook.com/c/16.html
- Prerequisite runs: Labs 00–04 (their checkpoints ARE the thing being evaluated)

**Why this lab exists:** loss curves are not evaluations. You build a mini-eval suite over ALL your
saved checkpoints, produce win-rate matrices from two judge arms (your Lab 03 RM and a local
Qwen3-4B LLM judge), then audit the judge itself: position / length / self-preference bias via
**order-swap flip rates**, and verdict-vs-ground-truth agreement on verifiable prompts.
Extension (absorbs old Lab 12): a generate → judge-filter synthetic preference loop and a
measurement of the bias the loop itself introduces.

**Gap-lab note:** there is no single repo module to copy here. The methodology reference is
`rlhf-book/code/rejection_sampling/diagnostics.py` (reward-vs-correctness diagnostics,
per-row win-rate vs baseline, `decidable_fraction` headroom framing) — reuse the *measurement
discipline*, not the code.

**Rules:** implement everything yourself before opening the reference. Every code cell below is a
TODO placeholder — no outputs are pre-computed.

## 0 · Setup

In [ ]:
# TODO(Lab11): imports, seed, paths, and a smoke-run budget (8 prompts x 2 checkpoints x both arms)
from starter import EvalConfig, load_prompt_set, build_head_to_head_pairs, run_checkpoint_suite  # noqa: F401
config = EvalConfig()
config

Checkpoint inventory: fill `config.checkpoint_paths` with your Labs 00–04(/06/07) artifacts and
`config.rm_checkpoint_path` with your Lab 03 RM. Labels become matrix row/column names.

## 1 · Fixed prompt set + head-to-head pairing

Frozen JSONL: `prompt_id`, `prompt`, `category` (`gsm8k_verifiable` | `open_ended` |
`instruction_following`). The matrix is only comparable across checkpoints evaluated on the
identical set — write down what breaks if you edit prompts mid-suite.

In [ ]:
# TODO(Lab11): assemble + freeze the prompt set; validate with load_prompt_set
# TODO(Lab11): enumerate comparison units with build_head_to_head_pairs; report the cell count
#              (#checkpoints^2 x #prompts) and judge-call cost BEFORE launching anything

In [ ]:
# TODO(Lab11): generate one greedy answer per checkpoint per prompt; store JSONL per checkpoint

## 2 · Two judge arms

Arm (a) — your Lab 03 Bradley-Terry RM (documented tie band; margins, not scale).
Arm (b) — `Qwen/Qwen3-4B` with a forced parseable output (`ANSWER: A|B|TIE`); unparseable is
counted, never silently a tie. Temperature 0; display order randomized (position-bias control).

In [ ]:
# TODO(Lab11): rm_judge_verdict — score both completions, decide verdict + margin
# TODO(Lab11): llm_judge_verdict — build judge prompt, parse, record order_shown
# TODO(Lab11): smoke-run both arms on 8 prompts; report unparseable rate

## 3 · Win-rate matrices (both arms)

In [ ]:
# TODO(Lab11): compute_win_rate_matrix per arm — document tie convention + diagonal;
#              un-flip order_shown='ba' records BEFORE aggregating
# TODO(Lab11): assert win_rates[i][j] + win_rates[j][i] == 1.0 on real data (ties convention)
# TODO(Lab11): heatmap both matrices; where do RM and LLM-judge rankings disagree?

## 4 · Judge-bias audit: order-swap flip rates

Flip = the verdict favors a DIFFERENT completion when presentation order swaps. Tie conflicts are
bucketed separately, never averaged away.

In [ ]:
# TODO(Lab11): order_swap_flip_rate — n_flips, flip_rate, direction split (first vs second)
# TODO(Lab11): flips vs verdict margin — do flips concentrate on near-ties?
# TODO(Lab11): position_bias_report  — P(first-position win), by model pair
# TODO(Lab11): length_bias_report    — P(longer wins) by gap bucket (<=/> 200 tokens)
# TODO(Lab11): self_preference_bias_report — lineage delta (judge shares lineage with 'sft'?)

## 5 · Verdicts vs verifiable ground truth

GSM8K-style exact match via YOUR Lab 06 verifier. `decidable_fraction` is a property of the
prompt set — report headroom separately from judge skill (diagnostics.py framing).

In [ ]:
# TODO(Lab11): verdict_vs_ground_truth — agreement on decidable prompts + confusion counts
# TODO(Lab11): per-arm agreement table (rm vs llm judge)

## 6 · Extension (absorbs old Lab 12): synthetic preferences

generate → judge-filter → measure the bias the loop itself introduces.

In [ ]:
# TODO(Lab11): generate_synthetic_preferences — K>=2 samples/prompt incl. a quality spread
# TODO(Lab11): judge_filter_preferences — kept (margin >= threshold) + rejected w/ reasons
# TODO(Lab11): measure_introduced_bias — length preference, position leakage, margin shift
# TODO(Lab11): which bias would a DPO model (Lab 04 loss) trained on the kept set inherit?

## 7 · Synthesis

- Which arm would you trust for the next training decision, and what is its measured failure mode?
- One paragraph: how the `decidable_fraction` of your prompt set caps every headline win-rate number.
- Artifacts: every stage appended one line to `runs/lab11_eval_judge/metrics.jsonl`.